In [1]:
from collections import Counter

from datasets import load_dataset
import pandas as pd
import numpy as np
import os

C:\Users\denve\miniconda3\envs\KG_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Downloading data
We are using a `milistu/AMAZON-Products-2023` dataset, which is based on `McAuley-Lab/Amazon-Reviews-2023`. It includes only products that were added on 2023 and also combines different categories in one dataset, which is exactly what we need. By default, it also includes embeddings for every product, while it would be really convenient to use them, it would restrict us to only using particular embedding model and may lead to potential anomalies, so we ignore them.

In [2]:
path = "../data/amazon_products.csv"

if not os.path.exists(path):
    dataset = load_dataset("milistu/AMAZON-Products-2023")
    df = dataset['train'].to_pandas()

    columns = ['parent_asin', 'date_first_available', 'title', 'description', 'main_category', 'average_rating', 'rating_number', 'price', 'features', 'details']

    df = df[columns]
    df.to_csv(path, columns=columns)

df = pd.read_csv(path, index_col=0, header=0)

df.head()

,parent_asin,date_first_available,title,description,main_category,average_rating,rating_number,price,features,details
0,B000044U2O,2023-04-29,Anomie & Bonhomie,Amazon.com\nFans of Scritti Politti's synth-po...,Digital Music,4.2,56.0,NaN,[],"{'Date First Available': 'April 29, 2023'}"
1,B0BT4CWWC9,2023-01-26,Sunshine On My Shoulders: The Best Of John Den...,"“Sunshine On My Shoulders” is a 2CD, 36-track ...",Digital Music,4.7,502.0,19.98,[],{'Package Dimensions': '5.55 x 4.92 x 0.51 inc...
2,B0BS4L5LP6,2023-01-11,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. ...,Digital Music,5.0,1.0,14.97,[],"{'Item Weight': '4 Ounces', 'Run time': '1 hou..."
3,B0BSPBBP89,2023-01-20,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music,4.8,34.0,12.99,[],{'Package Dimensions': '5.59 x 4.8 x 0.47 inch...
4,B0BT1YG8MV,2023-01-20,ΤΗΕ ΒΟΟΤLΕG SΕRΙΕS VοΙ. ᛐ7 ᛐ996-ᛐ997 FRΑԌΜΕΝΤՏ...,"EU Edition 2CD,\nDISC ONE - TIME OUT OF MIND [...",Digital Music,3.6,5.0,43.99,[],"{'Manufacturer': 'Columbia Records, Sony Music..."


# Data cleaning and processing
We start by cleaning text data from unexpected characters

In [3]:
from data_processing.data_cleaning import clean_text, convert_to_json


Tools and Home Improvement


In [4]:
# clean columns with text data
columns_to_clean = ['title', 'description', 'features', 'details']
df[columns_to_clean] = df[columns_to_clean].map(clean_text)

# replace empty strings with NaN for easier processing
df = df.replace("", np.nan)
df = df.replace("[]", np.nan)

# convert features values from strings to lists
df['features'] = df['features'].str.strip("[]{}")
df = df.replace("", np.nan)
df = df.replace("[]", np.nan)
df = df.replace("Null", np.nan)
df = df.replace("None", np.nan)
df = df.dropna(subset=['title', 'price', 'description', 'parent_asin'])
df = df.dropna(thresh=6)



# convert details to dictionaries
df['details'] = df['details'].map(convert_to_json)

# convert date from strings to actual date format
df['date_first_available'] = pd.to_datetime(df['date_first_available'])
df.head()

,parent_asin,date_first_available,title,description,main_category,average_rating,rating_number,price,features,details
1,B0BT4CWWC9,2023-01-26,Sunshine On My Shoulders: The Best Of John Den...,"Sunshine On My Shoulders is a 2CD, 36-track co...",Digital Music,4.7,502.0,19.98,NaN,{'Package Dimensions': '5.55 x 4.92 x 0.51 inc...
2,B0BS4L5LP6,2023-01-11,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. H...,Digital Music,5.0,1.0,14.97,NaN,"{'Item Weight': '4 Ounces', 'Run time': '1 hou..."
3,B0BSPBBP89,2023-01-20,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music,4.8,34.0,12.99,NaN,{'Package Dimensions': '5.59 x 4.8 x 0.47 inch...
5,B0BPT1369H,2023-02-14,HIGH DRAMA. CD SIGNED-ADAM LAMBERT,Renowned for re-imagining songs from American ...,Digital Music,4.7,207.0,23.35,NaN,"{'Language': 'English', 'Product Dimensions': ..."
6,B0BVYD326Y,2023-02-15,Hamilton Original Broadway Cast Recording (Exp...,Explicit version. 2015 two CD set. The origina...,Digital Music,4.6,56.0,21.86,NaN,{'Package Dimensions': '5.55 x 4.88 x 0.94 inc...


In [5]:
df.to_pickle("../data/amazon_products_cleaned.pkl")